# BME 603/606: Computational Methods for Biological Modeling & Simulation
## In-Class Activity: Modeling and Simulating Chemical Reactions

1. **Negative Autoregulation** — write the ODE, simulate it, and observe how a transcription factor represses its own production to reach a stable steady state.

2. **Phase Portraits** — visualize the trajectory of a system in phase space and interpret the vector field.

3. **The Bistable Toggle Switch** — simulate mutual repression between two proteins and observe bistability.

4. **Challenge (Bonus)** — extend the toggle switch into a three-gene **repressilator** and observe limit-cycle oscillations.

Let's get started!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Imports done!")

---
## Part 1: Negative Autoregulation

### Background: Why Negative Autoregulation?

One of the most common network motifs in biology is **negative autoregulation**: a transcription factor (TF) represses its own gene. Found in ~40% of *E. coli* transcription factors, this motif allows the cell to:

- **Reach a steady state faster** than a simple constitutive gene.
- **Buffer against noise** in upstream signals.
- **Maintain robust expression levels** despite fluctuations.

### The Mathematical Model

Let $[TF]$ be the concentration of the transcription factor. The rate of change is:

$$\frac{d[TF]}{dt} = k_s \cdot \frac{K^n}{K^n + [TF]^n} - d \cdot [TF]$$

where:
- **$k_s$** is the maximal production rate (nM/min).
- **$K$** is the repression threshold (nM) — the TF concentration at which production is half-maximal.
- **$n$** is the Hill coefficient — how cooperative (switch-like) the repression is.
- **$d$** is the degradation rate constant (min$^{-1}$).

The first term is a **Hill inhibition function**, $f_{\text{inh}}([TF]) = \frac{K^n}{K^n + [TF]^n}$, which approaches 1 when $[TF] \ll K$ (gene is on) and 0 when $[TF] \gg K$ (gene is off).

### Exercise 1: Simulate Negative Autoregulation

**Tasks:**
1. Complete the `negative_autoregulation` function below to define the ODE.
2. Use `solve_ivp` to simulate the system from $t = 0$ to $t = 30$ min, starting at $[TF]_0 = 0$.
3. Plot $[TF]$ vs. time.
4. Verify that the steady state matches the analytical prediction: set $\frac{d[TF]}{dt} = 0$ and solve for $[TF]_{ss}$. Print your analytical result and the numerical result next to each other.

**Parameters:** $k_s = 5.0$ nM/min, $d = 0.5$ min $^{-1}$, $K = 3.0$ nM, $n = 3$.

In [ ]:
# Parameters
ks, kd, K, n = 5.0, 0.5, 3.0, 3

def negative_autoregulation(t, y):
    """
    ODE for negative autoregulation.
    y[0] = [TF]
    """
    TF = y[0]
    # Your code here — compute repression and dTF/dt
    repression = None
    dTF_dt = None
    return [dTF_dt]

# Initial condition and time span
y0 = [0.0]
t_span = (0, 30)
t_eval = np.linspace(0, 30, 300)

# Your code here — run solve_ivp
# sol = ...

# Your code here — plot TF vs time

# Your code here — print analytical vs numerical steady state

---
## Part 2: Phase Portraits

### Background: What is a Phase Portrait?

A **phase portrait** plots the state variables against each other (e.g., $[TF]$ vs. $\frac{d[TF]}{dt}$, or $[A]$ vs. $[B]$ in a two-variable system) to visualize the **flow** of the system.

For a one-variable system, the phase portrait is simply $\frac{d[TF]}{dt}$ as a function of $[TF]$. The points where the curve crosses zero are **fixed points** (steady states).

**Stability:**
- If $\frac{d}{d[TF]}\left(\frac{d[TF]}{dt}\right) < 0$ at a fixed point, it is **stable** (the system is pulled back toward it).
- If the derivative is $> 0$, the fixed point is **unstable** (the system is pushed away).

### Exercise 2: Visualize the Phase Portrait for Negative Autoregulation

**Tasks:**
1. Create an array of $[TF]$ values from 0 to 15.
2. Compute $\frac{d[TF]}{dt}$ for each value using your ODE function.
3. Plot $\frac{d[TF]}{dt}$ vs. $[TF]$. Add a horizontal line at 0.
4. Mark the steady state (where the curve crosses 0) with a red dot.
5. **Interpret:** Is the fixed point stable or unstable? Verify by checking the sign of the derivative on either side of the fixed point.

**Hint:** You can compute the derivative at a given $TF$ by calling `negative_autoregulation(0, [tf])[0]`.

In [ ]:
# Your code here — create TF array, compute derivative, plot

# Your code here — mark steady state and interpret stability


---
## Part 3: The Bistable Toggle Switch

### Background: Cell Fate Decisions

A **toggle switch** consists of two transcription factors, A and B, that **mutually repress** each other. This motif is the basis of bistability in cell fate decisions: e.g. the λ phage lysis-lysogeny switch (see: https://pubmed.ncbi.nlm.nih.gov/16285866/) and stem cell differentiation rely on it.

### The Mathematical Model

$$\frac{d[A]}{dt} = k \cdot \frac{K^n}{K^n + [B]^n} - d \cdot [A]$$

$$\frac{d[B]}{dt} = k \cdot \frac{K^n}{K^n + [A]^n} - d \cdot [B]$$

where:
- **$k$** is the maximal production rate.
- **$K$** is the repression threshold.
- **$n$** is the Hill coefficient (cooperativity).
- **$d$** is the degradation rate.

Because A represses B and B represses A, the system has **two stable steady states**: one where A is high and B is low, and one where B is high and A is low. Which state the system settles into depends on the initial conditions.

### Exercise 3: Simulate the Toggle Switch

**Tasks:**
1. Complete the `toggle_switch` function below.
2. Simulate the system from $t = 0$ to $t = 60$ min for two different initial conditions:
   - **(Case 1):** $[A]_0 = 0.1$, $[B]_0 = 10$
   - **(Case 2):** $[A]_0 = 10$, $[B]_0 = 0.1$
3. Plot both cases on the same figure (A solid, B dashed).
4. **Interpret:** Which protein wins in each case? Does the system reach a stable steady state? Why or why not?

**Parameters:** $k = 5$ nM/min, $d = 0.5$ min$^{-1}$, $K = 3$ nM, $n = 3$.

**Hint:** Your function returns `[dA_dt, dB_dt]`. `y[0]` is A, `y[1]` is B.

In [ ]:
# Parameters
k, d, K, n = 5.0, 0.5, 3.0, 3

def toggle_switch(t, y):
    """
    ODEs for the mutual repression toggle switch.
    y[0] = [A], y[1] = [B]
    """
    A, B = y[0], y[1]
    # Your code here — compute repression of A by B, and of B by A
    dA_dt = None
    dB_dt = None
    return [dA_dt, dB_dt]

# Time span
t_span = (0, 60)
t_eval = np.linspace(0, 60, 600)

# Your code here — simulate Case 1 and Case 2

# Your code here — plot A and B for both cases

# Your code here — interpret the results

### Exercise 4: Phase Portrait of the Toggle Switch

For a two-variable system, the phase portrait plots $[A]$ on the x-axis and $[B]$ on the y-axis. The vector field $(\frac{d[A]}{dt}, \frac{d[B]}{dt})$ tells us the direction of flow at every point.

**Tasks:**
1. Create a grid of $(A, B)$ values (e.g., using `np.meshgrid` on ranges 0 to 10).
2. Compute $(\frac{d[A]}{dt}, \frac{d[B]}{dt})$ at each grid point.
3. Plot the vector field using `plt.quiver` or `plt.streamplot`.
4. Overlay the trajectories from Case 1 and Case 2 from Exercise 3.
5. **Interpret:** Where are the stable steady states? Why does the system go to one or the other depending on the initial condition?

**Hint:** `plt.streamplot` often looks cleaner than `quiver` for continuous flows.

In [ ]:
# Your code here — create A, B grid, compute vector field

# Your code here — plot vector field and trajectories

# Your code here — interpret stable steady states

---
## Part 4: Challenge Exercise (Bonus)

### Exercise 5: The Repressilator — A Limit Cycle Oscillator

**For bonus credit.** The toggle switch has two stable states. What happens if we connect **three** repressors in a cycle?

The **repressilator** (Elowitz & Leibler, 2000) is a synthetic gene circuit where:
- Protein A represses B,
- B represses C,
- C represses A.

With sufficient cooperativity ($n \geq 3$) and balanced parameters, this system exhibits **sustained oscillations** — a **limit cycle**.

### The Mathematical Model

$$\frac{d[A]}{dt} = k \cdot \frac{K^n}{K^n + [C]^n} - d \cdot [A]$$

$$\frac{d[B]}{dt} = k \cdot \frac{K^n}{K^n + [A]^n} - d \cdot [B]$$

$$\frac{d[C]}{dt} = k \cdot \frac{K^n}{K^n + [B]^n} - d \cdot [C]$$

**Tasks:**
1. Implement the `repressilator` ODE function.
2. Simulate from $t = 0$ to $t = 100$ min with initial conditions $[A]_0 = 1$, $[B]_0 = 0$, $[C]_0 = 0$.
3. Plot $[A]$, $[B]$, and $[C]$ vs. time. Do you see oscillations?
4. **Challenge:** Plot the 3D phase portrait (A vs. B vs. C). What shape does the trajectory trace out?
5. **Interpret:** Why does the toggle switch settle to a fixed point while the repressilator oscillates? What role does the **odd number of repressors** play?

**Parameters:** $k = 5$, $d = 0.5$, $K = 3$, $n = 3$.

**Hint:** Use `projection='3d'` with `fig.add_subplot(111, projection='3d')` for the 3D plot.

In [ ]:
def repressilator(t, y):
    """
    ODEs for the three-gene repressilator.
    y[0] = [A], y[1] = [B], y[2] = [C]
    """
    A, B, C = y[0], y[1], y[2]
    # Your code here — A is repressed by C, B by A, C by B
    dA_dt = None
    dB_dt = None
    dC_dt = None
    return [dA_dt, dB_dt, dC_dt]

# Your code here — simulate and plot time series

# Your code here — 3D phase portrait


**Your interpretation of the repressilator vs. the toggle switch:**

*Write your answer here. Why does an odd number of repressors lead to oscillation? What would happen with an even number?*

---
## Further Exploration

### The lac Operon

The *lac* operon in *E. coli* is a classic example of a bistable genetic switch, controlled by two regulators: CAP (activator) and LacI (repressor). See: [The lac Operon — Wikipedia](https://en.wikipedia.org/wiki/Lac_operon).

### The Repressilator Paper

Elowitz, M. B., & Leibler, S. (2000). A synthetic oscillatory network of transcriptional regulators. *Nature*, 403(6767), 335-338. [Read the original paper](https://www.nature.com/articles/35002125).

### BioModels Database

The [BioModels Database](https://www.ebi.ac.uk/biomodels/) contains hundreds of curated SBML models of signaling and gene regulatory networks. Downloading and exploring existing models is a great next step.

### References

- [Solve IVP — SciPy Docs](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html)
- [Hill Equation — Wikipedia](https://en.wikipedia.org/wiki/Hill_equation_(biochemistry))
- [Repressilator — Wikipedia](https://en.wikipedia.org/wiki/Repressilator)
- [Toggle Switch (Genetics) — Wikipedia](https://en.wikipedia.org/wiki/Toggle_switch_(genetics))